# **MEASURE evaluation pipeline**

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report
import json
from pathlib import Path
from utils.helpers import pdf_to_images, OrganizeLSOutput, FormateLSOutputReadingOrder
from utils.ocr_metrics import compute_metrics_wer_cer
from utils.ocr_output_norma import JsonSuryaNormalization, JsonTesseractNormalization, JsonEasyOCRNormalization, JsonDotsOCRNormalization, BaseJsonNormalization
from utils.ocr_output_norma import JsonAWSTextractNormalization
from utils.ocr_engines import EasyOCREngine, TesseractEngine, MistralOCREngine
from utils.reading_order_metrics import compute_ndld_dataframe
from utils.ocr_metrics import normalize_text
from utils.reading_order_metrics import prepare_refs_hyps, compute_bleu
from utils.layout_metrics import BuildLines, ComputeMAP
from utils.inferences import DonutInference

In [ ]:
resume_images_directory = "data/resumes_images"

## **Running the OCRs**

Surya

In [ ]:
!chmod +x ../utils/suryaOCR.sh
!bash ../utils/suryaOCR.sh /data/resumes_images output_dir/

EasyOCR

In [ ]:
easyocr_processor = EasyOCREngine(
    input_dir=resume_images_directory,
    output_dir="output_dir",
)

easyocr_processor.process_images()

Tesserract OCR

In [ ]:
tesseract_processor = TesseractEngine(
    input_dir=resume_images_directory,
    output_dir="output_dir")

tesseract_processor.process_images()

Mistral Basic OCR

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv() 

# this output is alreayd formmated in the way we need for the metrics, so no need to do any additional formatting
engine = MistralOCREngine(
    input_dir=resume_images_directory,
    api_key=os.getenv("MISTRAL_API_KEY")
)

* Dots OCR and Dots MOCR inferences were made by using Runpod
* AWS Textract `aws_textract_inference.py` file is the script we used for AWS Textract inference.

## **Ground-truth normalization**

In [ ]:
ground_truth_data = pd.read_csv("/data/ground_truth_LS_112.csv")

ground_truth_data = FormateLSOutputReadingOrder(ground_truth_data).process()

ground_truth_data = OrganizeLSOutput(
    ground_truth_data,
    transcription_column="ro_transcription",
    rename={"words": "words_ro"},
    images_directory=resume_images_directory
).process_all()

## **Formatting OCR outputs**

In [ ]:
# mistral basic ocr -------- (these results are already normalized, so we just read the csv)
predictions_data_mistral = pd.read_csv("/data/mistral_results.csv")

# surya ----------
normalizer_surya = JsonSuryaNormalization(images_directory=resume_images_directory)
predictions_data_surya = normalizer_surya.process_directory("/data/surya_output")

# tesseract ----------
normalizer_tesseract = JsonTesseractNormalization(images_directory=resume_images_directory)
predictions_data_tesseract = normalizer_tesseract.process_directory("/data/tesseract_output")

# easyocr ----------
normalizer_easyocr = JsonEasyOCRNormalization(images_directory=resume_images_directory)
predictions_data_easyocr = normalizer_easyocr.process_directory("/data/easyocr_output")

# dotsocr ----------
normalizer_dotsocr = JsonDotsOCRNormalization(images_directory=resume_images_directory)
predictions_data_dotsocr = normalizer_dotsocr.process_directory("/data/dotsocr_output")

# aws textract
normalizer_aws_textract = JsonAWSTextractNormalization(images_directory=resume_images_directory)
predictions_data_aws_textract = normalizer_aws_textract.process_directory("/data/aws_textract_output")

# dots MOCR ------
# same normalization as dotsocr since the output format is the same
predictions_data_dotmocr = normalizer_dotsocr.process_directory("/data/dots_mocr_output") 

## **Metrics computation**

### **Computing WER and CER**

OCR metrics

In [ ]:
results_bow = {}
results_standard = {}

models = {
    'surya': predictions_data_surya,
    'tesseract': predictions_data_tesseract, 
    'easyocr': predictions_data_easyocr,
    'dotsocr': predictions_data_dotsocr,
    'mistralai': predictions_data_mistral,
    'aws textract': predictions_data_aws_textract,
    'dots MOCR': predictions_data_dotmocr

}

for model_name, predictions_df in models.items():
    standard_results_df = compute_metrics_wer_cer(predictions_df, ground_truth_data, ground_truth_words_col='words_ro')
    results_standard[model_name] = standard_results_df

standard_summary_data = []
for model_name, results_df in results_standard.items():
    mean_wer = results_df['wer_score'].mean()
    mean_cer = results_df['cer_score'].mean()
    standard_summary_data.append({
        'Model': model_name,
        'Mean WER': mean_wer,
        'Mean CER': mean_cer
    })

standard_summary_table = pd.DataFrame(standard_summary_data)

In [ ]:
print("WER and CER with Reading Order")
standard_summary_table.sort_values('Mean WER').reset_index(drop=True)

### **Computing NDLD and BLEU**

Reading order metrics for OCRs

In [ ]:
models_config = [
    {'name': name, 'df': df, 'pred_col': 'words'}
    for name, df in models.items()
]

In [ ]:
word_level_results = []
for config in models_config:
    print(f"Computing WORD level NDLD for {config['name']}...")
    
    scores_df_word = compute_ndld_dataframe(
        ground_truth_data,
        config['df'],
        key="filename",
        gt_col="words_ro",
        pred_col=config['pred_col'],
        level="word",
        normalize_fn=lambda t: normalize_text(t, lowercase=True, remove_punct=True,
                                            remove_diacritics=True, dehyphenate=True)
    )
    
    word_level_results.append({
        'Model': config['name'],
        'Mean_Similarity': scores_df_word['similarity'].mean(),
        'Mean_NDLD': scores_df_word['ndld'].mean(),
    })

char_level_results = []
for config in models_config:
    print(f"Computing CHARACTER level NDLD for {config['name']}...")
    
    scores_df_char = compute_ndld_dataframe(
        ground_truth_data,
        config['df'],
        key="filename",
        gt_col="words_ro",
        pred_col=config['pred_col'],
        level="character",
        normalize_fn=lambda t: normalize_text(t, lowercase=True, remove_punct=False,
                                            remove_diacritics=False, dehyphenate=True)
    )
    
    char_level_results.append({
        'Model': config['name'],
        'Mean_Similarity': scores_df_char['similarity'].mean(),
        'Mean_NDLD': scores_df_char['ndld'].mean(),
    })

word_level_df = pd.DataFrame(word_level_results)
char_level_df = pd.DataFrame(char_level_results)

In [ ]:
print("NDLD at word level")
word_level_df

In [ ]:
page_bleu_rows = []
corpus_bleu_rows = []

for cfg in models_config:
    print(f"Scoring BLEU for {cfg['name']}...")

    hyps, refs = prepare_refs_hyps(
        ensayo_ro=ground_truth_data,
        pred_df=cfg['df'],
        pred_col=cfg['pred_col'],
        gt_filename_col="filename",
        gt_text_col="words_ro",
    )

    metrics = compute_bleu(hyps, refs)

    page_bleu_rows.append({
        "Model": cfg["name"],
        "Pages scored": int(metrics["pages_scored"]),
        "Avg Page BLEU": metrics["avg_page_bleu"],
    })

    corpus_bleu_rows.append({
        "Model": cfg["name"],
        "Corpus BLEU": metrics["corpus_bleu"],
        "P1": metrics["p1"],
        "P2": metrics["p2"],
        "P3": metrics["p3"],
        "P4": metrics["p4"],
        "Brevity Penalty": metrics["brevity_penalty"],
    })

page_bleu_df = (
    pd.DataFrame(page_bleu_rows)
      .sort_values("Avg Page BLEU", ascending=False)
      .reset_index(drop=True)
)

corpus_bleu_df = (
    pd.DataFrame(corpus_bleu_rows)
      .sort_values("Corpus BLEU", ascending=False)
      .reset_index(drop=True)
)

In [ ]:
page_bleu_df

In [ ]:
corpus_bleu_df

### **Computing mAP@IoU; mAP@0.5 and mAP@0.75**

Computing Mean Average Precision for Tesserract, AWS Textract, Tasyocr and Suryaocr bounding boxes

In [ ]:
predictions_data_aws_textract_eval = ComputeMAP.prepare_aws_textract_df(
    predictions_data_aws_textract
)

ground_truth_data = BuildLines.build_line_columns(
    ground_truth_data, 
    text_col='ro_transcription', 
    bbox_col='ro_bboxes', 
    y_tol=0.6
)

In [ ]:
tess_lines = BuildLines.build_line_columns(
    predictions_data_tesseract,
    text_col="words", bbox_col="bbox",
    input_mode="xyxy", 
    y_tol=0.6
)

easy_lines = BuildLines.build_line_columns(
    predictions_data_easyocr,
    text_col="text", bbox_col="bbox",
    input_mode="xyxy",
    y_tol=0.6
)

surya_lines = BuildLines.build_line_columns(
    predictions_data_surya,
    text_col="text_line", bbox_col="bbox",
    input_mode="xyxy",
    y_tol=0.6
)

aws_lines = BuildLines.build_line_columns(
    predictions_data_aws_textract_eval,
    text_col="text",
    bbox_col="bbox",
    input_mode="xywh",
    y_tol=0.6
)

In [ ]:
gt_lu = ComputeMAP.make_gt_lookup(
    ground_truth_data, 
    key_col="filename",
    gt_bbox_col="bbox_in_line", 
    w_col="width", 
    h_col="height"
)

tess_eval = ComputeMAP.add_metrics_to_pred_df(
    pred_df=tess_lines, 
    gt_lookup=gt_lu,
    key_col="filename", 
    pred_bbox_col="bbox_in_line", 
    confidence_col="confidence",
    thresholds=(0.5, 0.75), 
    prefix="tess"
)

easy_eval = ComputeMAP.add_metrics_to_pred_df(
    pred_df=easy_lines, 
    gt_lookup=gt_lu,
    key_col="filename", 
    pred_bbox_col="bbox_in_line", 
    confidence_col="confidence",
    thresholds=(0.5, 0.75), 
    prefix="easy"
)

surya_eval = ComputeMAP.add_metrics_to_pred_df(
    pred_df=surya_lines, 
    gt_lookup=gt_lu,
    key_col="filename", 
    pred_bbox_col="bbox_in_line", 
    confidence_col="confidence",
    thresholds=(0.5, 0.75), 
    prefix="surya"
)

aws_textract_eval = ComputeMAP.add_metrics_to_pred_df(
    pred_df=aws_lines, 
    gt_lookup=gt_lu,
    key_col="filename", 
    pred_bbox_col="bbox_in_line", 
    confidence_col="confidence",
    thresholds=(0.5, 0.75), 
    prefix="aws_textract"
)

tess_summary = ComputeMAP.summarize_pred_df(tess_eval, prefix="tess")
easy_summary = ComputeMAP.summarize_pred_df(easy_eval, prefix="easy")
surya_summary = ComputeMAP.summarize_pred_df(surya_eval, prefix="surya")
aws_summary = ComputeMAP.summarize_pred_df(aws_textract_eval, prefix="aws_textract")

In [ ]:
aws_summary

In [ ]:
tess_summary

In [ ]:
easy_summary

In [ ]:
surya_summary

### **End-to-end evaluation**

**Donut**

In [ ]:
from utils.inferences import DonutInference

In [ ]:
donut = DonutInference(ckpt="naver-clova-ix/donut-base")
results_donut = donut.infer_directory(resume_images_directory)
results_donut.head()

In [ ]:
word_level_results = []
scores_df_word = compute_ndld_dataframe(
    ground_truth_data,
    results_donut,
    key="filename",
    gt_col="words_ro",
    pred_col='words',
    level="word",
    normalize_fn=lambda t: normalize_text(t, lowercase=True, remove_punct=True,
                                            remove_diacritics=True, dehyphenate=True))
    
word_level_results.append({
    'Model': 'donut',
    'Mean_Similarity': scores_df_word['similarity'].mean(),
    'Mean_NDLD': scores_df_word['ndld'].mean()})

char_level_results = []
scores_df_char = compute_ndld_dataframe(
    ground_truth_data,
    results_donut,
    key="filename",
    gt_col="words_ro",
    pred_col='words',
    level="character",
    normalize_fn=lambda t: normalize_text(t, lowercase=True, remove_punct=False,
                                            remove_diacritics=False, dehyphenate=True))
    
char_level_results.append({
    'Model': 'donut',
    'Mean_Similarity': scores_df_char['similarity'].mean(),
    'Mean_NDLD': scores_df_char['ndld'].mean()})

word_level_df = pd.DataFrame(word_level_results)
char_level_df = pd.DataFrame(char_level_results)

page_bleu_rows = []
corpus_bleu_rows = []

hyps, refs = prepare_refs_hyps(
    ground_truth_data,
    results_donut,
    pred_col="words",
    gt_filename_col="filename",
    gt_text_col="words_ro")

metrics = compute_bleu(hyps, refs)
page_bleu_rows.append({
    "Model": "donut",
    "Pages scored": int(metrics["pages_scored"]),
    "Avg Page BLEU": metrics["avg_page_bleu"]})

corpus_bleu_rows.append({
    "Model": "donut",
    "Corpus BLEU": metrics["corpus_bleu"],
    "P1": metrics["p1"],
    "P2": metrics["p2"],
    "P3": metrics["p3"],
    "P4": metrics["p4"],
    "Brevity Penalty": metrics["brevity_penalty"]})

page_bleu_df = (
    pd.DataFrame(page_bleu_rows)
      .sort_values("Avg Page BLEU", ascending=False)
      .reset_index(drop=True)
)

corpus_bleu_df = (
    pd.DataFrame(corpus_bleu_rows)
      .sort_values("Corpus BLEU", ascending=False)
      .reset_index(drop=True)
)

In [ ]:
word_level_df

In [ ]:
corpus_bleu_df

In [ ]:
page_bleu_df

**LayoutReader**

* LayoutReader with EasyOCR bboxes
* LayoutReader with SuryaOCR bboxes
* LayoutReader with Tesseract bboxes

In [ ]:
from utils.inferences import LayoutReaderInference

lr = LayoutReaderInference(device="cpu")

df_surya = lr.run(
    input_dir="/Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/send_to_ministral",
    external_ocr_frames=[predictions_data_surya],
    text_column_pred="text_line",
)

df_easy = lr.run(
    input_dir="/Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/send_to_ministral",
    external_ocr_frames=[predictions_data_easyocr],
    text_column_pred="text",
)

df_tess = lr.run(
    input_dir="/Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/send_to_ministral",
    external_ocr_frames=[predictions_data_tesseract],
    text_column_pred="words",
)

In [ ]:
models_config = [
    {"name": "Surya",     "df_lr": df_surya, "pred_col_for_metrics": "words_LR_order"},
    {"name": "EasyOCR",   "df_lr": df_easy,  "pred_col_for_metrics": "words_LR_order"},
    {"name": "Tesseract", "df_lr": df_tess,  "pred_col_for_metrics": "words_LR_order"},
]

word_level_rows = []
char_level_rows = []

for cfg in models_config:
    print(f"Computing WORD level NDLD for {cfg['name']}...")
    scores_df_word = compute_ndld_dataframe(
        ground_truth_data,
        cfg["df_lr"],
        key="filename",
        gt_col="words_ro",
        pred_col=cfg["pred_col_for_metrics"],
        level="word",
        normalize_fn=lambda t: normalize_text(
            t, lowercase=True, remove_punct=True, remove_diacritics=True, dehyphenate=True
        ),
    )
    word_level_rows.append({
        "Model": cfg["name"],
        "Word Mean Similarity": scores_df_word["similarity"].mean(),
        "Word Mean NDLD":       scores_df_word["ndld"].mean(),
    })

for cfg in models_config:
    print(f"Computing CHARACTER level NDLD for {cfg['name']}...")
    scores_df_char = compute_ndld_dataframe(
        ground_truth_data,
        cfg["df_lr"],
        key="filename",
        gt_col="words_ro",
        pred_col=cfg["pred_col_for_metrics"],
        level="character",
        normalize_fn=lambda t: normalize_text(
            t, lowercase=True, remove_punct=False, remove_diacritics=False, dehyphenate=True
        ),
    )
    char_level_rows.append({
        "Model": cfg["name"],
        "Char Mean Similarity": scores_df_char["similarity"].mean(),
        "Char Mean NDLD":       scores_df_char["ndld"].mean(),
    })

word_level_df = pd.DataFrame(word_level_rows)
char_level_df = pd.DataFrame(char_level_rows)

bleu_rows = []
for cfg in models_config:
    print(f"Computing BLEU for {cfg['name']}...")
    hyps, refs = prepare_refs_hyps(
        ground_truth_data,
        cfg["df_lr"],
        pred_col=cfg["pred_col_for_metrics"],
        gt_filename_col="filename",
        gt_text_col="words_ro",
    )
    metrics = compute_bleu(hyps, refs)
    bleu_rows.append({
        "Model": cfg["name"],
        "Pages Scored":    int(metrics.get("pages_scored", 0)),
        "Avg Page BLEU":   metrics.get("avg_page_bleu", float("nan")),
        "Corpus BLEU":     metrics.get("corpus_bleu", float("nan")),
        "P1":              metrics.get("p1", float("nan")),
        "P2":              metrics.get("p2", float("nan")),
        "P3":              metrics.get("p3", float("nan")),
        "P4":              metrics.get("p4", float("nan")),
        "Brevity Penalty": metrics.get("brevity_penalty", float("nan")),
    })

bleu_df = pd.DataFrame(bleu_rows)
bleu_df

In [ ]:
word_level_df

### **NER evaluation**

**LayoutLMv3**

In [ ]:
from transformers import AutoProcessor, AutoModelForTokenClassification
from utils.inferences import LayoutLmv3Inference, LiLTInference

In [ ]:
def label_maps(df, sections_col="ro_section"):
    labels = sorted(set(df.iloc[0][sections_col]))
    if "O" not in labels:
        labels.insert(0, "O")
    id2label = {i: lab for i, lab in enumerate(labels)}
    label2id = {lab: i for i, lab in id2label.items()}
    return id2label, label2id, labels

id2label, label2id, labels = label_maps(
    ground_truth_data,
    sections_col="ro_section",
)

print(id2label)

In [ ]:
model_name = "microsoft/layoutlmv3-base"
processor = AutoProcessor.from_pretrained(model_name, apply_ocr=False)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id)

infer = LayoutLmv3Inference(
    df_ground_truth=ground_truth_data,
    images_base_path=resume_images_directory,
    model_name=model_name,
    sections_col="ro_section",
    text_col="ro_transcription",
    bbox_col="ro_bboxes",
    id2label=id2label,
    label2id=label2id,
    labels=labels,
    processor=processor,
    model=model)

results = infer.run()
results_layoutlmv3 = pd.DataFrame(results)
results_layoutlmv3.head()

In [ ]:
all_gt = sum(results_layoutlmv3["gt_sections"].tolist(), [])
all_pred = sum(results_layoutlmv3["pred_labels"].tolist(), [])
print(classification_report(all_gt, all_pred))

**LiLT**

In [ ]:
model_name = "SCUT-DLVCLab/lilt-roberta-en-base"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

infer_lilt = LiLTInference(
    df_ground_truth=ground_truth_data,
    images_base_path=resume_images_directory,
    model_name=model_name,
    sections_col="ro_section",
    text_col="ro_transcription",
    bbox_col="ro_bboxes",
    id2label=id2label,
    label2id=label2id,
    labels=labels,     
    processor=processor,    
    model=model,            
)

results_lilt = infer_lilt.run()

In [ ]:
all_gt = sum(results_lilt["gt_sections"].tolist(), [])
all_pred = sum(results_lilt["pred_labels"].tolist(), [])
print(classification_report(all_gt, all_pred))